# Phase 3 — UNet
Run this notebook on Kaggle. After training, download the `results/` folder and `checkpoints/best_model.pth` and share with Claude.

In [ ]:
!pip install -q torchmetrics scikit-image

In [ ]:
import os, csv, json, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from torchmetrics.image import StructuralSimilarityIndexMeasure as SSIM
from torchmetrics.image import MultiScaleStructuralSimilarityIndexMeasure as MSSSIM
from skimage.metrics import peak_signal_noise_ratio as psnr_sk
from skimage.metrics import structural_similarity as ssim_sk

os.makedirs('/kaggle/working/results',     exist_ok=True)
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
DATA_ROOT      = '/kaggle/input/datasets/nachiketpatil0105/spc-data/train/'
LEARNING_RATE  = 1e-4
NUM_EPOCHS     = 150
GRAD_CLIP_NORM = 1.0
PATCH_SIZE     = 512
AUG_PROB       = 0.5
W_CHARB, W_MSSSIM, W_VGG = 0.25, 0.50, 0.25
LOG_CSV = '/kaggle/working/results/training_log.csv'

In [ ]:
def unpack_spc(npy_path):
    data = np.load(npy_path, mmap_mode='r')[-128:]
    data = np.unpackbits(data, axis=2)
    data = data.reshape(128, 800, 800, 3)
    data = np.transpose(data, (1, 2, 0, 3))
    data = data.reshape(800, 800, 384)
    data = data / (data.max() + 1e-8)
    return data.transpose(2, 0, 1).astype(np.float32)

class SPCDataset(Dataset):
    def __init__(self, npy_paths, png_paths, augment=False):
        self.npy_paths = npy_paths
        self.png_paths = png_paths
        self.augment   = augment

    def __len__(self): return len(self.npy_paths)

    def _augment(self, x, img):
        if random.random() < AUG_PROB:
            x = np.flip(x, axis=2).copy(); img = np.flip(img, axis=2).copy()
        if random.random() < AUG_PROB:
            x = np.flip(x, axis=1).copy(); img = np.flip(img, axis=1).copy()
        k = random.randint(0, 3)
        if k > 0:
            x = np.rot90(x, k, axes=(1,2)).copy(); img = np.rot90(img, k, axes=(1,2)).copy()
        return x, img

    def _crop(self, x, img):
        _, H, W = x.shape
        top = random.randint(0, H - PATCH_SIZE)
        left = random.randint(0, W - PATCH_SIZE)
        return x[:, top:top+PATCH_SIZE, left:left+PATCH_SIZE], img[:, top:top+PATCH_SIZE, left:left+PATCH_SIZE]

    def __getitem__(self, idx):
        x   = unpack_spc(self.npy_paths[idx])
        img = np.array(Image.open(self.png_paths[idx])).astype(np.float32) / 255.0
        img = img.transpose(2, 0, 1)
        if self.augment:
            x, img = self._augment(x, img)
            x, img = self._crop(x, img)
        return torch.from_numpy(x).float(), torch.from_numpy(img).float()

In [ ]:
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps
    def forward(self, pred, target):
        diff = pred - target
        return torch.mean(torch.sqrt(diff * diff + self.eps ** 2))

class VGGPerceptualLoss(nn.Module):
    LAYER_WEIGHTS = {'relu1_2': 0.2, 'relu2_2': 0.3, 'relu3_3': 0.5}
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        features = list(vgg.features.children())
        self.slice1 = nn.Sequential(*features[:5]).eval()
        self.slice2 = nn.Sequential(*features[5:10]).eval()
        self.slice3 = nn.Sequential(*features[10:17]).eval()
        for p in self.parameters(): p.requires_grad = False
        self.l1 = nn.L1Loss()
        self.w  = self.LAYER_WEIGHTS
    def forward(self, pred, target):
        f1_p = self.slice1(pred);   f1_t = self.slice1(target)
        f2_p = self.slice2(f1_p);   f2_t = self.slice2(f1_t)
        f3_p = self.slice3(f2_p);   f3_t = self.slice3(f2_t)
        return (self.w['relu1_2']*self.l1(f1_p,f1_t) +
                self.w['relu2_2']*self.l1(f2_p,f2_t) +
                self.w['relu3_3']*self.l1(f3_p,f3_t))

class UNetBasic(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = self._block(384, 128); self.pool1 = nn.MaxPool2d(2)
        self.enc2 = self._block(128, 256); self.pool2 = nn.MaxPool2d(2)
        self.enc3 = self._block(256, 512); self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = self._block(512, 1024)
        self.up1  = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec1 = self._block(512+512, 512)
        self.up2  = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec2 = self._block(256+256, 256)
        self.up3  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = self._block(128+128, 128)
        self.out  = nn.Conv2d(128, 3, 1)

    def _block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        e1=self.enc1(x);  p1=self.pool1(e1)
        e2=self.enc2(p1); p2=self.pool2(e2)
        e3=self.enc3(p2); p3=self.pool3(e3)
        b=self.bottleneck(p3)
        u1=self.up1(b);  d1=self.dec1(torch.cat([u1,e3],dim=1))
        u2=self.up2(d1); d2=self.dec2(torch.cat([u2,e2],dim=1))
        u3=self.up3(d2); d3=self.dec3(torch.cat([u3,e1],dim=1))
        return self.out(d3)

In [ ]:
path      = Path(DATA_ROOT)
npy_files = sorted(path.rglob('*.npy'))
png_files = sorted(path.rglob('*.png'))
n         = len(npy_files)
n_test    = max(1, int(0.10 * n))
n_val     = max(1, int(0.10 * n))
n_train   = n - n_val - n_test

train_X, train_y = npy_files[:n_train],              png_files[:n_train]
val_X,   val_y   = npy_files[n_train:n_train+n_val],  png_files[n_train:n_train+n_val]
test_X,  test_y  = npy_files[n_train+n_val:],         png_files[n_train+n_val:]
print(f'Train: {len(train_X)}  Val: {len(val_X)}  Test: {len(test_X)}')

train_loader = DataLoader(SPCDataset(train_X, train_y, augment=True),  batch_size=1, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(SPCDataset(val_X,   val_y),                  batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(SPCDataset(test_X,  test_y),                 batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
model     = UNetBasic().to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler    = GradScaler('cuda')
charb_fn  = CharbonnierLoss(eps=1e-3)
vgg_fn    = VGGPerceptualLoss().to(device)
ssim_fn   = SSIM(data_range=1.0).to(device)
msssim_fn = MSSSIM(data_range=1.0, kernel_size=11).to(device)

with open(LOG_CSV, 'w', newline='') as f:
    csv.writer(f).writerow(['epoch','lr','train_loss','train_ssim','train_msssim','train_psnr','val_loss','val_ssim','val_msssim','val_psnr'])

def compute_psnr(pred, target):
    mse = torch.mean((pred - target)**2).item()
    return float('inf') if mse == 0 else 10.0 * np.log10(1.0 / mse)

print('All ready.')

In [ ]:
best_val_ssim = -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Train ---
    model.train()
    t_loss = t_ssim = t_msssim = t_psnr = 0.0
    for x, target in train_loader:
        x, target = x.to(device), target.to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            pred = model(x); pred_c = torch.clamp(pred, 0, 1)
            ms   = msssim_fn(pred_c, target)
            loss = W_CHARB*charb_fn(pred,target) + W_MSSSIM*(1-ms) + W_VGG*vgg_fn(pred_c,target)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer); scaler.update()
        with torch.no_grad():
            t_ssim += ssim_fn(pred_c, target).item()
            t_psnr += compute_psnr(pred_c, target)
            t_msssim += ms.item()
        t_loss += loss.item()
    n_tr = len(train_loader)
    t_loss/=n_tr; t_ssim/=n_tr; t_msssim/=n_tr; t_psnr/=n_tr

    # --- Validate ---
    model.eval()
    v_loss = v_ssim = v_msssim = v_psnr = 0.0
    with torch.no_grad():
        for x, target in val_loader:
            x, target = x.to(device), target.to(device)
            with autocast('cuda'):
                pred = model(x); pred_c = torch.clamp(pred, 0, 1)
                ms   = msssim_fn(pred_c, target)
                loss = W_CHARB*charb_fn(pred,target) + W_MSSSIM*(1-ms) + W_VGG*vgg_fn(pred_c,target)
            v_loss+=loss.item(); v_ssim+=ssim_fn(pred_c,target).item()
            v_msssim+=ms.item(); v_psnr+=compute_psnr(pred_c,target)
    n_vl=max(len(val_loader),1)
    v_loss/=n_vl; v_ssim/=n_vl; v_msssim/=n_vl; v_psnr/=n_vl

    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    print(f'Epoch {epoch:3d}/{NUM_EPOCHS}  LR:{lr:.2e}  '
          f'| Train Loss:{t_loss:.4f} SSIM:{t_ssim:.4f} PSNR:{t_psnr:.2f}  '
          f'| Val Loss:{v_loss:.4f} SSIM:{v_ssim:.4f} PSNR:{v_psnr:.2f}')

    with open(LOG_CSV,'a',newline='') as f:
        csv.writer(f).writerow([epoch,f'{lr:.2e}',f'{t_loss:.4f}',f'{t_ssim:.4f}',f'{t_msssim:.4f}',f'{t_psnr:.2f}',f'{v_loss:.4f}',f'{v_ssim:.4f}',f'{v_msssim:.4f}',f'{v_psnr:.2f}'])

    if v_ssim > best_val_ssim:
        best_val_ssim = v_ssim
        torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),'val_ssim':v_ssim},
                   '/kaggle/working/checkpoints/best_model.pth')
        print(f'  ✓ New best val SSIM: {best_val_ssim:.4f}')

print(f'\nDone. Best val SSIM: {best_val_ssim:.4f}')

In [ ]:
# Training curves
epochs_log=[]; t_loss_log=[]; v_loss_log=[]; t_ssim_log=[]; v_ssim_log=[]; t_psnr_log=[]; v_psnr_log=[]
with open(LOG_CSV) as f:
    for row in csv.DictReader(f):
        epochs_log.append(int(row['epoch']))
        t_loss_log.append(float(row['train_loss'])); v_loss_log.append(float(row['val_loss']))
        t_ssim_log.append(float(row['train_ssim'])); v_ssim_log.append(float(row['val_ssim']))
        t_psnr_log.append(float(row['train_psnr'])); v_psnr_log.append(float(row['val_psnr']))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('UNet — Training & Validation Curves', fontsize=14, fontweight='bold')
for ax, (t, v, title) in zip(axes, [(t_loss_log,v_loss_log,'Loss'),(t_ssim_log,v_ssim_log,'SSIM'),(t_psnr_log,v_psnr_log,'PSNR (dB)')]):
    ax.plot(epochs_log, t, label='Train'); ax.plot(epochs_log, v, label='Val')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/results/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/training_curves.png')

In [ ]:
# Load best model and evaluate on test set
ckpt = torch.load('/kaggle/working/checkpoints/best_model.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded epoch {ckpt['epoch']}  val SSIM {ckpt['val_ssim']:.4f}")

model.eval()
results = []

with torch.no_grad():
    for i, (x, target) in enumerate(test_loader):
        x, target = x.to(device), target.to(device)
        with autocast('cuda'):
            pred = torch.clamp(model(x), 0, 1)

        pred_np   = pred.squeeze().cpu().float().numpy().transpose(1,2,0)
        target_np = target.squeeze().cpu().float().numpy().transpose(1,2,0)
        input_np  = x.squeeze().cpu().float().numpy()[:3].transpose(1,2,0)

        p = psnr_sk(target_np, pred_np, data_range=1.0)
        s = ssim_sk(target_np, pred_np, channel_axis=2, data_range=1.0)
        scene = Path(test_y[i]).parent.name
        results.append({'sample':i,'scene':scene,'PSNR':float(p),'SSIM':float(s)})
        print(f'Sample {i} ({scene})  PSNR={p:.2f} dB  SSIM={s:.4f}')

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle(f'UNet Reconstruction — {scene}', fontsize=13, fontweight='bold')
        for ax, (img, title, sub) in zip(axes, [
            (input_np,  'SPC Input (first 3 ch)', ''),
            (pred_np,   'UNet Prediction',        f'PSNR={p:.2f} dB  SSIM={s:.4f}'),
            (target_np, 'Ground Truth',           ''),
        ]):
            ax.imshow(np.clip(img,0,1)); ax.set_title(title,fontsize=11,fontweight='bold')
            ax.set_xlabel(sub,fontsize=9); ax.axis('off')
        plt.tight_layout()
        plt.savefig(f'/kaggle/working/results/comparison_sample{i}_{scene}.png', dpi=150, bbox_inches='tight')
        plt.show(); plt.close()

avg_psnr = sum(r['PSNR'] for r in results)/len(results)
avg_ssim = sum(r['SSIM'] for r in results)/len(results)
print(f'\nAverage  PSNR={avg_psnr:.2f} dB  SSIM={avg_ssim:.4f}')

with open('/kaggle/working/results/metrics.json','w') as f:
    json.dump({'test_results':results,'avg_psnr':avg_psnr,'avg_ssim':avg_ssim},f,indent=2)
print('Saved → results/metrics.json')
print('\nDownload the entire results/ folder and share with Claude.')